# 자연어처리(NLP) 수업 정리

여러 강의 노트북(`1`, `5`, `6`, `6_1`, `6_2`, `7`, `7_1`, `7_2`, `7_3`)에서
**자연어처리(NLP)** 관련 내용만 모아 주제 순서대로 재구성한 노트북입니다.

**목차**
1. 토큰화(Tokenization) 기초
2. 말뭉치 / 서브워드 토크나이저 (SentencePiece, WordPiece)
3. N-gram, TF-IDF
4. Word2Vec (Skip-gram) 직접 구현
5. Word2Vec / FastText (gensim)
6. RNN / LSTM 기초
7. RNN/LSTM 기반 문장 분류 (NSMC 감성분류)
8. 사전학습 임베딩(Word2Vec)을 활용한 LSTM 분류기
9. CNN 기반 문장 분류 (TextCNN)
10. 텍스트 데이터 증강 (nlpaug)
11. Transformer (Positional Encoding, Seq2Seq 번역)
12. GPT-2 텍스트 생성 / BERT 감성분류
13. BART 뉴스 요약
14. ELECTRA 감성분류 (한국어)
15. T5 요약


## 1. 토큰화(Tokenization) 기초

어절/음절/자모 단위 분리부터 KoNLPy(Okt, Kkma), NLTK, spaCy를 이용한 토큰화·품사 태깅까지 다룹니다.

### 1-1. 어절 단위 / 음절 단위 / 자모 단위 토큰화

In [ ]:
review = '현실과 구분 불가능한 cg. 시각적 즐거움은 최고! 더불어 ost는 더더욱 최고!!'
tokenized = review.split()
print(tokenized)


In [ ]:
review = '현실과 구분 불가능한 cg. 시각적 즐거움은 최고! 더불어 ost는 더더욱 최고!!'
tokenized = list(review)
print(tokenized)

In [ ]:
from jamo import h2j, j2hcj
review = '현실과 구분 불가능한 cg. 시각적 즐거움은 최고! 더불어 ost는 더더욱 최고!!'
decomposed = j2hcj(h2j(review))
tokenized = list(decomposed)
print(tokenized)

### 1-2. KoNLPy(Okt, Kkma)를 이용한 한국어 형태소 분석

In [ ]:
from konlpy.tag import Okt

okt = Okt()

sentence = '무엇이든 상상할 수 있는 사람은 무엇이든 만들어 낼 수 있다.'

# 명사
nouns = okt.nouns(sentence)
# 구
phrases = okt.phrases(sentence)
# 형태소
morphs = okt.morphs(sentence)
# 품사
pos = okt.pos(sentence)

print(nouns)
print(phrases)
print(morphs)
print(pos)





In [ ]:
from konlpy.tag import Kkma

kkma = Kkma()

sentence = '무엇이든 상상할 수 있는 사람은 무엇이든 만들어 낼 수 있다.'

# 명사
nouns = kkma.nouns(sentence)
# 구
sentences = kkma.sentences(sentence)
# 형태소
morphs = kkma.morphs(sentence)
# 품사
pos = kkma.pos(sentence)

print(nouns)
print(sentences)
print(morphs)
print(pos)


### 1-3. NLTK를 이용한 영어 토큰화 및 품사 태깅

In [ ]:
import nltk

nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
from nltk import tokenize

sentence = 'Those who can imagine anything, can create the impossible'
word_tokens = tokenize.word_tokenize(sentence)
sent_tokens = tokenize.sent_tokenize(sentence)

print(word_tokens)
print(sent_tokens)

In [ ]:
from nltk import tag
from nltk import tokenize

sentence = 'Those who can imagine anything, can create the impossible'
word_tokens = tokenize.word_tokenize(sentence)
sent_tokens = tokenize.sent_tokenize(sentence)
pos = tag.pos_tag(word_tokens)

print(pos)

### 1-4. spaCy를 이용한 토큰화 및 품사 태깅

In [ ]:
import sys
!{sys.executable} -m spacy download en_core_web_sm

In [ ]:
import spacy

nlp = spacy.load('en_core_web_sm')
sentence = 'Those who can imagine anything, can create the impossible'
doc = nlp(sentence)

for token in doc:
    print(token.pos, token.tag_, token.text)

## 2. 말뭉치 구축 & 서브워드 토크나이저

Korpora로 말뭉치를 내려받아 저장한 뒤, SentencePiece(BPE)와 WordPiece 토크나이저를 직접 학습시켜봅니다.

### 2-1. Korpora 말뭉치 불러오기 및 저장

In [ ]:
from Korpora import Korpora

corpus = Korpora.load('korean_petitions')
dataset = corpus.train
petition = dataset[0]

print(petition.begin)
import os
os.makedirs('./dataset', exist_ok=True)

In [ ]:
petitions = corpus.get_all_texts()
with open('./datasets/corpos.txt','w',encoding='utf-8') as f:
    for petition in petitions:
        f.write(petition + '\n')

### 2-2. SentencePiece(BPE) 토크나이저 학습 및 사용

In [ ]:
from sentencepiece import SentencePieceTrainer

SentencePieceTrainer.Train(
    '--input=./datasets/corpus.txt\
    --model_prefix=./models/petition.bpe\
        --vocab_size=8000 model_type=bpe'
)

In [ ]:
from sentencepiece import SentencePieceProcessor

tokenizer = SentencePieceProcessor()
tokenizer.load('./models/petition.bpe.model')

sentence = "안녕하세요. 토크나이저가 잘 학습되었군요!."
sentences = ['이렇게 입력값을 리스트로 받아서', '쉽게 토크나이저를 사용할 수 있답니다.']

tokenized_sentence = tokenizer.encode_as_pieces(sentence)
tokenized_sentences = tokenizer.encode_as_pieces(sentences)
print(tokenized_sentence)
print(tokenized_sentences)

encoded_sentence = tokenizer.encode_as_ids(sentence)
encoded_sentences = tokenizer.encode_as_ids(sentences)
print(encoded_sentence)
print(encoded_sentences)

decoded_ids = tokenizer.decode_ids(encoded_sentence)
decoded_pieces = tokenizer.decode_pieces(tokenized_sentences)
print(decoded_ids)
print(decoded_pieces)

In [ ]:
vocab = {idx: tokenizer.id_to_piece(idx) for idx in range(tokenizer.get_piece_size())}
print(list(vocab.items())[:5])
print(len(vocab))

### 2-3. WordPiece 토크나이저 학습 및 사용

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.normalizers import Sequence, NFD, Lowercase
from tokenizers.pre_tokenizers import Whitespace

tokenizer = Tokenizer(WordPiece())
tokenizer.normalizer = Sequence([NFD(), Lowercase()])
tokenizer.pre_tokenizer = Whitespace()

# tokenizer.train(['./datasets/corpus.txt'])
# tokenizer.save('./models/petition_wordpiece.json')

from itertools import islice

with open('./datasets/corpus.txt', encoding = 'utf-8') as f:
    tokenizer.train_from_iterator(islice(f, 100_000))
tokenizer.save('./models/petition_wordpiece.json')

In [ ]:
from tokenizers import Tokenizer
from tokenizers.decoders import WordPiece as WordPieceDecoder

tokenizer = Tokenizer.from_file('./models/petition_wordpiece.json')
tokenizer.decoder = WordPieceDecoder()

sentence = "안녕하세요. 토크나이저가 잘 학습되었군요!."
sentences = ['이렇게 입력값을 리스트로 받아서', '쉽게 토크나이저를 사용할 수 있답니다.']

encoded_sentence = tokenizer.encode(sentence)
encoded_sentences = tokenizer.encode_batch(sentences)

print(type(encoded_sentence))
print(encoded_sentence.tokens)
print([enc.tokens for enc in encoded_sentences])
print(encoded_sentence.ids)
print([enc.ids for enc in encoded_sentences])
print(tokenizer.decode(encoded_sentence.ids))

## 3. N-gram과 TF-IDF

### 3-1. N-gram 직접 구현 & NLTK ngrams

In [ ]:
import nltk

def ngrams(sentence, n):
    words = sentence.split()
    ngrams = zip(*[words[i:] for i in range(n)])
    return list(ngrams)

sentence = '안녕하세요. 만나서 진심으로 반가워요.'

unigram = ngrams(sentence,1)
bigram= ngrams(sentence,2)
trigram=ngrams(sentence,3)

print(unigram)
print(bigram)
print(trigram)

unigram = nltk.ngrams(sentence.split(), 1)
bigram = nltk.ngrams(sentence.split(),2)
trigram = nltk.ngrams(sentence.split(),3)

print(list(unigram))
print(list(bigram))
print(list(trigram))

### 3-2. TF-IDF 벡터화 (scikit-learn)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = [
    'That movie is famous movie',
    'I like that actor',
    "I don't like that actor",
    
]

tfidf_vectorizer = TfidfVectorizer()
tfidf_vectorizer.fit(corpus)
tfidf_matrix = tfidf_vectorizer.transform(corpus)

print(tfidf_matrix.toarray())
print(tfidf_vectorizer.vocabulary_)

## 4. Word2Vec (Skip-gram) 직접 구현

NSMC(네이버 영화 리뷰) 말뭉치를 형태소 분석한 뒤, Skip-gram 모델을 PyTorch로 직접 구현하고 학습해 단어 임베딩을 얻습니다.

### 4-1. Skip-gram 모델 정의

In [ ]:
import torch.nn as nn
class VanillaSkipgram(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings= vocab_size,
            embedding_dim= embedding_dim
        )
        self.linear = nn.Linear(
            in_features = embedding_dim,
            out_features= vocab_size
        )
    def forward(self, input_ids):
        embeddings = self.embedding(input_ids)
        output = self.linear(embeddings)
        return output

### 4-2. NSMC 말뭉치 로드 및 형태소 분석

In [ ]:
import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt


corpus = Korpora.load('nsmc')

corpus = pd.DataFrame(corpus.test)

In [ ]:
tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus.text]
print(tokens[:3])

### 4-3. 단어 사전(vocab) 구축

In [ ]:
from collections import Counter
def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab

vocab = build_vocab(corpus=tokens, n_vocab=5000,
                    special_tokens = ['<unk>'])
token_to_id = {token:idx for idx, token in enumerate(vocab)}
id_to_token = {idx:token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

### 4-4. 중심 단어-주변 단어 쌍 생성

In [ ]:
def get_word_pairs(tokens, window_size):
    pairs = []
    for sentence in tokens:
        sentence_length = len(sentence)
        for idx, center_word in enumerate(sentence):
            window_start = max(0, idx-window_size)
            window_end = min(sentence_length,idx+window_size+1)
            center_word = sentence[idx]
            context_words = sentence[window_start:idx] + sentence[idx+1:window_end]
            for context_word in context_words:
                pairs.append([center_word,context_word])

    return pairs
word_pairs = get_word_pairs(tokens,window_size = 2)
print(word_pairs[:5])

In [ ]:
def get_index_pairs(word_pairs, token_to_id):
    pairs = []
    unk_index = token_to_id['<unk>']
    for word_pair in word_pairs:
        center_word, context_word = word_pair
        center_index = token_to_id.get(center_word, unk_index)
        context_index = token_to_id.get(context_word, unk_index)
        pairs.append([center_index, context_index])
    return pairs

index_pairs = get_index_pairs(word_pairs, token_to_id)
print(index_pairs[:5])
print(len(vocab))

### 4-5. 데이터셋/데이터로더 구성 및 모델 학습

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

index_pairs=torch.tensor(index_pairs)
center_indexes = index_pairs[:,0]
context_indexes = index_pairs[:,1]

dataset = TensorDataset(center_indexes, context_indexes)
dataloader = DataLoader(dataset,batch_size=32,shuffle=True)

In [ ]:
import torch.optim as optim
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
word2vec = VanillaSkipgram(vocab_size=len(token_to_id),
                          embedding_dim=128).to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.SGD(word2vec.parameters(),lr=0.1)

In [ ]:
for epoch in range(10):
    cost = 0.0
    for input_ids, target_ids in dataloader:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)

        logits = word2vec(input_ids)
        loss = criterion(logits,target_ids)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        cost += loss.item()
    cost = cost / len(dataloader)
    print(epoch+1,cost)

### 4-6. 학습된 임베딩 확인 및 코사인 유사도로 유사 단어 찾기

In [ ]:
token_to_embedding = dict()
embedding_matrix = word2vec.embedding.weight.detach().cpu().numpy()

for word,embedding in zip(vocab, embedding_matrix):
    token_to_embedding[word] = embedding
index = 30
token = vocab[index]
token_embedding = token_to_embedding[token]
print(f"Token: {token}")
print(f"Embedding: {token_embedding}")

In [ ]:
import numpy as np
from numpy.linalg import norm

def cosine_similarity(a,b):
    cosine = np.dot(b,a) / (norm(b, axis=1)*norm(a))
    return cosine

def top_n_index(cosine_matrix,n):
    closest_indexes = cosine_matrix.argsort()[::-1]
    top_n = closest_indexes[1:n+1]
    return top_n
cosine_matrix = cosine_similarity(token_embedding, embedding_matrix)
top_n = top_n_index(cosine_matrix,n=5)

for index in top_n:
    print(id_to_token[index], cosine_matrix[index])

## 5. Word2Vec / FastText (gensim 라이브러리 활용)

gensim을 이용해 Word2Vec, FastText 모델을 학습하고 저장/로드, 유사도 계산, OOV(미등록 단어) 벡터 추론까지 실습합니다.

### 5-1. gensim Word2Vec 학습

In [ ]:
import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt

corpus = Korpora.load('nsmc')
corpus = pd.DataFrame(corpus.test)


In [ ]:
tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus.text]

In [ ]:
from gensim.models import Word2Vec

word2vec= Word2Vec(
    sentences = tokens,
    vector_size= 128,
    window = 5,
    min_count = 1,
    sg = 1,
    epochs= 3,
    max_final_vocab = 10000

)

In [ ]:
word2vec.save('./models/word2vec.model')
word2vec = Word2Vec.load('./models/word2vec.model')

In [ ]:
word = '연기'
print(word2vec.wv[word])
print(word2vec.wv.most_similar(word,topn=5))
print(word2vec.wv.similarity(w1=word, w2 = '연기력'))

### 5-2. gensim FastText 학습 (subword 기반 OOV 처리)

In [ ]:
from Korpora import Korpora

corpus = Korpora.load('kornli')
corpus_texts = corpus.get_all_texts() + corpus.get_all_pairs()
tokens = [sentence.split() for sentence in corpus_texts]

print(tokens[:3])

In [ ]:
from gensim.models import FastText

fastText = FastText(
    sentences = tokens,
    vector_size = 128,
    window=5,
    min_count=5,
    sg=1,
    max_final_vocab= 20000,
    epochs = 3,
    min_n=2,
    max_n = 6

)

In [ ]:
oov_token = '사랑해요'
oov_vector = fastText.wv[oov_token]

print(oov_token in fastText.wv.index_to_key)
print(fastText.wv.most_similar(oov_vector, topn=5))

## 6. RNN / LSTM 기초

PyTorch의 `nn.RNN`, `nn.LSTM` 레이어의 입출력 shape을 직접 확인합니다.

In [ ]:
import torch
import torch.nn as nn

device = 'cuda' if torch.cuda.is_available else 'cpu'
print(device)

In [ ]:
input_size=128
output_size=256
num_layers=3
bidirectional=True

model = nn.RNN(
    input_size = input_size
    , hidden_size= output_size,
    num_layers= num_layers,
    nonlinearity='tanh',
    batch_first = True,
    bidirectional = bidirectional
).to(device)

batch_size=4
sequence_len =6

inputs = torch.randn(batch_size, sequence_len, input_size).to(device)
h_0 = torch.rand(num_layers*(int(bidirectional)+1), batch_size,
                 output_size).to(device)
outputs, hidden = model(inputs,h_0)

print(outputs.shape)
print(hidden.shape)
print(outputs.device)

In [ ]:
input_size = 128
output_size = 256
num_layers = 3
bidirectional = True
proj_size = 64

model = nn.LSTM(
    input_size= input_size,
    hidden_size = output_size,
    num_layers = num_layers,
    batch_first= True,
    bidirectional= bidirectional,
    proj_size= proj_size
)

batch_size=4
sequence_len=6

inputs=torch.randn(batch_size, sequence_len, input_size)
h_0 = torch.rand(
    num_layers * (int(bidirectional) + 1),
    batch_size,
    proj_size if proj_size > 0 else output_size 
)

c_0 = torch.rand(num_layers * (int(bidirectional)+1), batch_size, output_size)

outputs, (h_n, c_n) = model(inputs,(h_0,c_0))

print(outputs.shape)
print(h_n.shape)
print(c_n.shape)



## 7. RNN/LSTM 기반 문장 분류 (NSMC 감성분류)

임베딩 → RNN/LSTM → 분류기로 이어지는 문장 분류 모델을 직접 구현하고, NSMC 감성분류 데이터로 학습/평가합니다.

### 7-1. SentenceClassifier 모델 정의

In [ ]:
import torch.nn as nn

class SentenceClassifier(nn.Module):
    def __init__(self,
                 n_vocab,
                 hidden_dim,
                 embedding_dim,
                 n_layers,
                 dropout=0.5,
                 bidirectional=True,
                 model_type='lstm'
                 ):
        super().__init__()
        self.embedding = nn.Embedding(
                num_embeddings=n_vocab,
                embedding_dim=embedding_dim,
                padding_idx = 0
            )
        if model_type =='rnn':
            self.model=nn.RNN(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers = n_layers,
                bidirectional = bidirectional,
                dropout = dropout,
                batch_first = True,
            )
        elif model_type =='lstm':
            self.model = nn.LSTM(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers = n_layers,
                bidirectional=bidirectional,
                dropout=dropout,
                batch_first=True,
            )

        if bidirectional:
            self.classifier = nn.Linear(hidden_dim*2,1)
        else:
            self.classifier = nn.Linear(hidden_dim,1)
        self.dropout=nn.Dropout(dropout)
            

    def forward(self,inputs):
        embeddings = self.embedding(inputs)
        output, _= self.model(embeddings)
        last_output = output[:,-1,:]
        last_output = self.dropout(last_output)
        logits=self.classifier(last_output)
        return logits
        

### 7-2. 데이터 준비 (train/test 분리)

In [ ]:
import pandas as pd
from Korpora import Korpora

corpus = Korpora.load('nsmc')
corpus_df = pd.DataFrame(corpus.test)

In [ ]:
train = corpus_df.sample(frac=0.9, random_state=42)
test = corpus_df.drop(train.index)

print(train.head().to_markdown())
print(len(train))
print(len(test))

### 7-3. 형태소 분석 및 단어 사전 구축

In [ ]:
from konlpy.tag import Okt
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab

tokenizer = Okt()
train_tokens = [tokenizer.morphs(review) for review in train.text]
test_tokens = [tokenizer.morphs(review) for review in test.text]

vocab = build_vocab(corpus= train_tokens, n_vocab=5000,
                    special_tokens=['<pad>','<unk>'])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}



### 7-4. 정수 인코딩 및 패딩

In [ ]:
import numpy as np

def pad_sequence(sequences, max_length, pad_value):
    result = list()
    for sequence in sequences:
        sequence = sequence[:max_length]
        pad_length = max_length - len(sequence)
        padded_sequence = sequence + [pad_value] * pad_length
        result.append(padded_sequence)

    return np.asarray(result)

unk_id = token_to_id['<unk>']
train_ids = [
    [token_to_id.get(token,unk_id) for token in review] for review in train_tokens]

test_ids = [[token_to_id.get(token, unk_id) for token in review] for review in test_tokens]

max_length=32
pad_id = token_to_id['<pad>']
train_ids = pad_sequence(train_ids, max_length, pad_id)
test_ids = pad_sequence(test_ids, max_length, pad_id)

print(train_ids[0])
print(test_ids[0])



### 7-5. 데이터로더 구성

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

train_ids = torch.tensor(train_ids)
test_ids = torch.tensor(test_ids)

train_labels = torch.tensor(train.label.values, dtype=torch.float32)
test_labels = torch.tensor(test.label.values, dtype=torch.float32)

train_dataset = TensorDataset(train_ids, train_labels)
test_dataset = TensorDataset(test_ids, test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16,shuffle=False)

print(len(train_loader))
print(len(test_loader))

### 7-6. 모델/손실함수/옵티마이저 정의

In [ ]:
import torch.optim as optim

n_vocab = len(token_to_id)
hidden_dim = 64
embedding_dim = 128
n_layers = 2

classifier = SentenceClassifier(
    n_vocab = n_vocab, hidden_dim = hidden_dim, embedding_dim= embedding_dim,
    n_layers= n_layers).to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.RMSprop((classifier.parameters()))

### 7-7. 학습 및 평가

In [ ]:
def train(model, datasets,criterion,optimizer,device,interval):
    model.train()
    losses = list()

    for step,(input_ids,labels) in enumerate(datasets):
        input_ids = input_ids.to(device)
        labels = labels.to(device).unsqueeze(1)

        logits = model(input_ids)
        loss = criterion(logits, labels)
        losses.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % interval ==0:
            print(step, np.mean(losses))

def test(model,datasets,criterion, device):
    model.eval()
    losses = list()
    corrects = list()
    for step, (input_ids,labels) in enumerate(datasets):
        input_ids = input_ids.to(device)
        labels = labels.to(device).unsqueeze(1)

        logits = model(input_ids)
        loss = criterion(logits, labels)
        losses.append(loss.item())
        yhat = torch.sigmoid(logits) > .5
        corrects.extend(
            torch.eq(yhat, labels).cpu().tolist()
        )
    print(np.mean(losses), np.mean(corrects))

epochs = 5
interval = 500

for epoch in range(epochs):
    train(classifier, train_loader, criterion, optimizer,device,interval)
    test(classifier,test_loader,criterion,device)

### 7-8. 학습된 임베딩 확인

In [ ]:
token_to_embedding = dict()
embedding_matrix = classifier.embedding.weight.detach().cpu().numpy()

for word,emb in zip(vocab,embedding_matrix):
    token_to_embedding[word] = emb

token=vocab[1000]
print(token, token_to_embedding[token])

## 8. 사전학습(Word2Vec) 임베딩을 활용한 LSTM 분류기

앞서 학습한 Word2Vec 임베딩을 초기값으로 사용하는 `SentenceClassifier`로 NSMC 감성분류 성능을 개선합니다.

### 8-1. 사전학습 임베딩을 지원하는 SentenceClassifier 정의

In [ ]:
import torch.nn as nn
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

class SentenceClassifier(nn.Module):
    def __init__(self,
                 n_vocab,
                 hidden_dim,
                 embedding_dim,
                 n_layers,
                 dropout=0.5,
                 bidirectional=True,
                 model_type='lstm',
                 pretrained_embedding=None
                 ):
        super().__init__()

        if pretrained_embedding is not None:
            self.embedding = nn.Embedding(
                torch.tensor(pretrained_embedding
                             ,dtype=torch.float32)
            )


        else: 
            self.embedding = nn.Embedding(
                num_embeddings=n_vocab,
                embedding_dim=embedding_dim,
                padding_idx = 0
            )
        if model_type =='rnn':
            self.model=nn.RNN(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers = n_layers,
                bidirectional = bidirectional,
                dropout = dropout,
                batch_first = True,
            )
        elif model_type =='lstm':
            self.model = nn.LSTM(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers = n_layers,
                bidirectional=bidirectional,
                dropout=dropout,
                batch_first=True,
            )

        if bidirectional:
            self.classifier = nn.Linear(hidden_dim*2,1)
        else:
            self.classifier = nn.Linear(hidden_dim,1)
        self.dropout=nn.Dropout(dropout)
            

    def forward(self,inputs):
        embeddings = self.embedding(inputs)
        output, _= self.model(embeddings)
        last_output = output[:,-1,:]
        last_output = self.dropout(last_output)
        logits=self.classifier(last_output)
        return logits
        

### 8-2. 데이터 준비 및 형태소 분석 / 단어 사전 구축

In [ ]:
import pandas as pd
from Korpora import Korpora

corpus = Korpora.load('nsmc')
corpus_df= pd.DataFrame(corpus.test)

In [ ]:
import pandas as pd
from Korpora import Korpora

corpus = Korpora.load('nsmc')
corpus_df = pd.DataFrame(corpus.test)

train = corpus_df.sample(frac=0.9, random_state=42)
test = corpus_df.drop(train.index)

print(train.head().to_markdown())
print(len(train))
print(len(test))

In [ ]:
import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt

tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus_df.text]
print(tokens[:3])
from collections import Counter
def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab

vocab = build_vocab(corpus=tokens, n_vocab=5000,
                    special_tokens = ['<unk>'])
token_to_id = {token:idx for idx, token in enumerate(vocab)}
id_to_token = {idx:token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

In [ ]:
from konlpy.tag import Okt
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab

tokenizer = Okt()
train_tokens = [tokenizer.morphs(review) for review in train.text]
test_tokens = [tokenizer.morphs(review) for review in test.text]

vocab = build_vocab(corpus= train_tokens, n_vocab=5000,
                    special_tokens=['<pad>','<unk>'])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}



### 8-3. 정수 인코딩·패딩 및 데이터로더 구성

In [ ]:
import numpy as np

def pad_sequence(sequences, max_length, pad_value):
    result = list()
    for sequence in sequences:
        sequence = sequence[:max_length]
        pad_length = max_length - len(sequence)
        padded_sequence = sequence + [pad_value] * pad_length
        result.append(padded_sequence)

    return np.asarray(result)

unk_id = token_to_id['<unk>']
train_ids = [
    [token_to_id.get(token,unk_id) for token in review] for review in train_tokens]

test_ids = [[token_to_id.get(token, unk_id) for token in review] for review in test_tokens]

max_length=32
pad_id = token_to_id['<pad>']
train_ids = pad_sequence(train_ids, max_length, pad_id)
test_ids = pad_sequence(test_ids, max_length, pad_id)

print(train_ids[0])
print(test_ids[0])



In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

train_ids = torch.tensor(train_ids)
test_ids = torch.tensor(test_ids)

train_labels = torch.tensor(train.label.values, dtype=torch.float32)
test_labels = torch.tensor(test.label.values, dtype=torch.float32)

train_dataset = TensorDataset(train_ids, train_labels)
test_dataset = TensorDataset(test_ids, test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16,shuffle=False)

print(len(train_loader))
print(len(test_loader))

In [ ]:
n_vocab = len(token_to_id)
hidden_dim = 64
embedding_dim = 128
n_layers = 2

### 8-4. 저장된 Word2Vec으로 임베딩 초기화

In [ ]:
from gensim.models import Word2Vec

word2vec = Word2Vec.load('./models/word2vec.model')
init_embeddings = np.zeros((n_vocab, embedding_dim))

for index, token in id_to_token.items():
    if token not in ['<pad>', '<unk>']:
        init_embeddings[index] = word2vec.wv[token]

embedding_layer = nn.Embedding.from_pretrained(
    torch.tensor(init_embeddings, dtype=torch.float32)
)



### 8-5. 학습/평가 함수 정의 및 학습

In [ ]:
def train(model, datasets,criterion,optimizer,device,interval):
    model.train()
    losses = list()

    for step,(input_ids,labels) in enumerate(datasets):
        input_ids = input_ids.to(device)
        labels = labels.to(device).unsqueeze(1)

        logits = model(input_ids)
        loss = criterion(logits, labels)
        losses.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % interval ==0:
            print(step, np.mean(losses))

def test(model,datasets,criterion, device):
    model.eval()
    losses = list()
    corrects = list()
    for step, (input_ids,labels) in enumerate(datasets):
        input_ids = input_ids.to(device)
        labels = labels.to(device).unsqueeze(1)

        logits = model(input_ids)
        loss = criterion(logits, labels)
        losses.append(loss.item())
        yhat = torch.sigmoid(logits) > .5
        corrects.extend(
            torch.eq(yhat, labels).cpu().tolist()
        )
    print(np.mean(losses), np.mean(corrects))

In [ ]:
import torch.optim as optim

classifier = SentenceClassifier(
                 n_vocab=n_vocab,
            hidden_dim=hidden_dim,
                 embedding_dim=embedding_dim,
                 n_layers=n_layers,
                 dropout=0.5,
                 bidirectional=True,
                 model_type='lstm',
                 pretrained_embedding=init_embeddings
                 ).to(device)

criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.RMSprop(classifier.parameters(), lr=0.001)
epochs = 5
interval = 500

for epoch in range(epochs):
    train(classifier, train_loader, criterion, optimizer,device,interval )
    test(classifier, test_loader, criterion, device)
    print(f"Epoch {epoch+1} completed.")

## 9. CNN 기반 문장 분류 (TextCNN)

사전학습 임베딩 위에 여러 크기의 1D 합성곱(Conv1d) 필터를 적용하는 TextCNN 구조로 NSMC 감성분류를 수행합니다.

> 원본 노트북에는 문법 오류가 있던 초안 셀과, 이를 수정한 최종본이 함께 있었는데 여기서는 **수정 완료된 최종 버전**만 남겼습니다.

### 9-1. TextCNN 기반 SentenceClassifier 정의

In [ ]:
import torch.nn as nn
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

class SentenceClassifier(nn.Module):
    def __init__(self, pretrained_embedding, filter_sizes, max_length, dropout=0.5):
        super().__init__()
        
        # [수정 1] 괄호 닫기 정상화
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(pretrained_embedding, dtype=torch.float32)
        )
        
        embedding_dim = self.embedding.weight.shape[1]
        conv = []
        
        for size in filter_sizes:
            # [수정 2] for문 내부 들여쓰기 적용
            conv.append(
                nn.Sequential(
                    nn.Conv1d(
                        in_channels=embedding_dim,
                        out_channels=1,
                        kernel_size=size
                    ), # [수정 3] Conv1d 끝에 콤마(,) 추가
                    nn.ReLU(),
                    # [수정 4] max_length - size - 1 이 아닌 + 1 이어야 차원이 맞습니다.
                    nn.MaxPool1d(kernel_size=max_length - size + 1)
                )
            )
            
        self.conv_filters = nn.ModuleList(conv)

        output_size = len(filter_sizes)
        # [수정 5] output.size (오타) -> output_size 로 수정
        self.pre_classifier = nn.Linear(output_size, output_size)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(output_size, 1)

    def forward(self, inputs):
        # [수정 6] forward 함수 내부 들여쓰기 적용
        embeddings = self.embedding(inputs)
        embeddings = embeddings.permute(0, 2, 1)
        
        # [수정 7] cov (오타) -> conv 로 수정
        conv_outputs = [conv(embeddings) for conv in self.conv_filters]
        
        # [수정 8] eonv_outputs (오타) -> conv_outputs 로 수정 및 변수명 겹침(conv -> out) 방지
        concat_outputs = torch.cat([out.squeeze(-1) for out in conv_outputs], dim=1)

        logits = self.pre_classifier(concat_outputs)
        logits = self.dropout(logits)
        logits = self.classifier(logits)
        return logits

### 9-2. 데이터 준비 및 형태소 분석

In [ ]:
import pandas as pd
from Korpora import Korpora

corpus = Korpora.load('nsmc')

corpus_df = pd.DataFrame(corpus.test)

In [ ]:
from konlpy.tag import Okt
tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus_df.text]
print(tokens[:3])

In [ ]:
train = corpus_df.sample(frac=0.9, random_state=42)
test = corpus_df.drop(train.index)

print(train.head().to_markdown())
print(len(train))
print(len(test))

### 9-3. 단어 사전 구축, 정수 인코딩·패딩

In [ ]:
from konlpy.tag import Okt
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab

tokenizer = Okt()
train_tokens = [tokenizer.morphs(review) for review in train.text]
test_tokens = [tokenizer.morphs(review) for review in test.text]

vocab = build_vocab(corpus= train_tokens, n_vocab=5000,
                    special_tokens=['<pad>','<unk>'])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}



In [ ]:
import numpy as np

def pad_sequence(sequences, max_length, pad_value):
    result = list()
    for sequence in sequences:
        sequence = sequence[:max_length]
        pad_length = max_length - len(sequence)
        padded_sequence = sequence + [pad_value] * pad_length
        result.append(padded_sequence)

    return np.asarray(result)

unk_id = token_to_id['<unk>']
train_ids = [
    [token_to_id.get(token,unk_id) for token in review] for review in train_tokens]

test_ids = [[token_to_id.get(token, unk_id) for token in review] for review in test_tokens]

max_length=32
pad_id = token_to_id['<pad>']
train_ids = pad_sequence(train_ids, max_length, pad_id)
test_ids = pad_sequence(test_ids, max_length, pad_id)

print(train_ids[0])
print(test_ids[0])



### 9-4. 데이터로더 구성

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

train_ids = torch.tensor(train_ids)
test_ids = torch.tensor(test_ids)

train_labels = torch.tensor(train.label.values, dtype=torch.float32)
test_labels = torch.tensor(test.label.values, dtype=torch.float32)

train_dataset = TensorDataset(train_ids, train_labels)
test_dataset = TensorDataset(test_ids, test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16,shuffle=False)

print(len(train_loader))
print(len(test_loader))

### 9-5. 사전학습 임베딩 로드 및 모델/옵티마이저 정의

In [ ]:
from gensim.models import Word2Vec

n_vocab = len(token_to_id)
embedding_dim=128

word2vec = Word2Vec.load('./models/word2vec.model')
init_embeddings = np.zeros((n_vocab, embedding_dim))

for index, token in id_to_token.items():
    if token not in ['<pad>', '<unk>']:
        init_embeddings[index] = word2vec.wv[token]

embedding_layer = nn.Embedding.from_pretrained(
    torch.tensor(init_embeddings, dtype=torch.float32)
)



In [ ]:
from torch import optim

filter_sizes = [3,3,4,4,5,5]

classifier = SentenceClassifier(pretrained_embedding=init_embeddings,
                                filter_sizes=filter_sizes,max_length=max_length).to(device)

criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.RMSprop(classifier.parameters(),lr=0.001)

### 9-6. 학습/평가 함수 정의 및 학습

In [ ]:
def train(model, datasets,criterion,optimizer,device,interval):
    model.train()
    losses = list()

    for step,(input_ids,labels) in enumerate(datasets):
        input_ids = input_ids.to(device)
        labels = labels.to(device).unsqueeze(1)

        logits = model(input_ids)
        loss = criterion(logits, labels)
        losses.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % interval ==0:
            print(step, np.mean(losses))

def test(model,datasets,criterion, device):
    model.eval()
    losses = list()
    corrects = list()
    for step, (input_ids,labels) in enumerate(datasets):
        input_ids = input_ids.to(device)
        labels = labels.to(device).unsqueeze(1)

        logits = model(input_ids)
        loss = criterion(logits, labels)
        losses.append(loss.item())
        yhat = torch.sigmoid(logits) > .5
        corrects.extend(
            torch.eq(yhat, labels).cpu().tolist()
        )
    print(np.mean(losses), np.mean(corrects))

In [ ]:
epochs = 5
interval = 500

for epoch in range(epochs):
    train(classifier, train_loader,criterion,optimizer,device,interval)
    test(classifier, test_loader, criterion, device)

## 10. 텍스트 데이터 증강 (nlpaug)

`nlpaug` 라이브러리로 문맥 기반 단어 삽입, 문자 삭제, 단어 swap, 동의어 치환, 예약어 보존, 역번역(Back-translation) 등 다양한 텍스트 증강 기법을 실습합니다.

### 10-1. NLTK 리소스 다운로드

In [ ]:
import nltk


In [ ]:
for resource in ['wordnet', 'omw-1.4', 'averaged_perceptron_tagger'
                 ,'averaged_perceptron_tagger_eng']:
    nltk.download(resource, quiet=True)

### 10-2. 문맥 기반 단어 삽입 증강 (BERT 활용)

In [ ]:
import nlpaug.augmenter.word as naw
texts = [
    'Those who can imagine anything, can create the impossible.',
    'We can only see a short distance ahead, but we can see plenty there that need to be done',
    'If a machine is expected to be in falliable, it cannot also be intelligent'
]

aug = naw.ContextualWordEmbsAug(model_path= 'bert-base-uncased',
                                action='insert', device='cpu')
augmented_texts = aug.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)

### 10-3. 문자 삭제 증강

In [ ]:
import nlpaug.augmenter.char as nac

texts = [
    'Those who can imagine anything, can create the impossible.',
    'We can only see a short distance ahead, but we can see plenty there that need to be done',
    'If a machine is expected to be in falliable, it cannot also be intelligent'
]

aug = nac.RandomCharAug(action='delete')
augmented_texts = aug.augment(text)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)

### 10-4. 단어 swap 증강

In [ ]:
import nlpaug.augmenter.word as naw

texts = [
    'Those who can imagine anything, can create the impossible.',
    'We can only see a short distance ahead, but we can see plenty there that need to be done',
    'If a machine is expected to be in falliable, it cannot also be intelligent'
]

aug = naw.RandomWordAug(action='swap')
augmented_texts = aug.augment(text)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)

### 10-5. 동의어(WordNet) 치환 증강

In [ ]:
import nlpaug.augmenter.word as naw

texts = [
    'Those who can imagine anything, can create the impossible.',
    'We can only see a short distance ahead, but we can see plenty there that need to be done',
    'If a machine is expected to be in falliable, it cannot also be intelligent'
]

aug = naw.SynonymAug(aug_src='wordnet')
augmented_texts = aug.augment(text)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)

### 10-6. 예약어(reserved words) 보존 증강

In [ ]:
import nlpaug.augmenter.word as naw

texts = [
    'Those who can imagine anything, can create the impossible.',
    'We can only see a short distance ahead, but we can see plenty there that need to be done',
    'If a machine is expected to be in falliable, it cannot also be intelligent'
]

reserved_tokens = [
    [
        "can","can't","cannot","could"
    ]
]

reserved_aug = naw.ReservedAug(reserved_tokens= reserved_tokens)
# aug = naw.SynonymAug(aug_src='wordnet')
augmented_texts = reserved_aug.augment(text)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)

### 10-7. 역번역(Back-translation) 증강

In [ ]:
import nlpaug.augmenter.word as naw

texts = [
    'Those who can imagine anything, can create the impossible.',
    'We can only see a short distance ahead, but we can see plenty there that need to be done',
    'If a machine is expected to be in falliable, it cannot also be intelligent'
]

back_translation = naw.BackTranslationAug(
    from_model_name = 'facebook/wmt19-en-de',
    to_model_name = 'facebook/wmt19-de-en',
    device = 'cpu'
)

augmented_texts = back_translation.augment(texts)
for text,augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)

## 11. Transformer로 기계번역 (Seq2Seq, 독일어→영어)

Positional Encoding부터 Multi30k 데이터셋 기반 토크나이저/사전 구축, `nn.Transformer`를 이용한 Seq2Seq 번역 모델 구현, 학습, greedy decoding을 이용한 번역까지 실습합니다.

### 11-1. Positional Encoding 구현 및 시각화

In [ ]:
import math
import torch
from torch import nn
import matplotlib.pyplot as plt

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0)/d_model)
        )
        pe = torch.zeros(max_len, 1, d_model)
        pe[:,0,0::2] = torch.sin(position*div_term)
        pe[:,0, 1::2] = torch.cos(position*div_term)
        self.register_buffer("pe",pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0)]
        return self.dropout(x)

encoding = PositionalEncoding(d_model=128, max_len=50)

plt.pcolormesh(encoding.pe.numpy().squeeze(), cmap='RdBu')
plt.xlabel('Embedding Dimension')
plt.xlim((0, 128))
plt.ylabel('Position')
plt.colorbar()
plt.show()

### 11-2. Multi30k 번역 데이터셋 확인

In [ ]:
import torch
from torchtext.datasets import Multi30k
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
print(torch.__version__)
print(torch.cuda.get_device_name(0))
from torchtext.datasets import Multi30k
train_iter = Multi30k(split='train', language_pair=('de', 'en'))
first = next(iter(train_iter))
print("데이터 샘플:", first)

### 11-3. 토크나이저 정의

In [ ]:
def generate_tokens(text_iter, language):
    langauge_index = {SRC_LANG: 0, TGT_LANG: 1}

    for text in text_iter:
        yield token_transform[language](text[langauge_index[language]])

SRC_LANG = 'de'
TGT_LANG = 'en'

UNK_IDX, PAD_IDX, BOS_IDX, EOS_IDX = 0,1,2,3
special_symbols = ['<unk>','<pad>','<bos>','<eos>']

token_transform = {
    SRC_LANG : get_tokenizer('spacy', language= 'de_core_news_sm'),
    TGT_LANG : get_tokenizer('spacy', language='en_core_web_sm')
}

print(token_transform)




### 11-4. 단어 사전(vocab) 구축

In [ ]:
vocab_transform = {}
for language in [SRC_LANG, TGT_LANG]:
    train_iter = Multi30k(split='train', language_pair=(SRC_LANG, TGT_LANG))
    vocab_transform[language] = build_vocab_from_iterator(
        generate_tokens(train_iter, language),min_freq=1,specials=special_symbols,special_first=True)

for language in [SRC_LANG, TGT_LANG]:
    vocab_transform[language].set_default_index(UNK_IDX)

print(vocab_transform)

### 11-5. Positional Encoding / Token Embedding / Seq2Seq Transformer 모델 정의

In [ ]:
import math
import torch
from torch import nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0)/d_model)
        )
        pe = torch.zeros(max_len, 1, d_model)
        pe[:,0,0::2] = torch.sin(position*div_term)
        pe[:,0, 1::2] = torch.cos(position*div_term)
        self.register_buffer("pe",pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0)]
        return self.dropout(x)

In [ ]:
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, emb_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size)
        self.emb_size = emb_size

    def forward(self, tokens):
        return self.embedding(tokens.long()) * math.sqrt(self.emb_size)

    

In [ ]:
class Seq2SeqTransformer(nn.Module):
    def __init__(self,num_encoder_layers, num_decoder_layers, emb_size,max_len,
                 nhead, src_vocab_size, tgt_vocab_size, dim_feedforward,
                 dropout=0.1):
        super().__init__()
        self.src_tok_emb = TokenEmbedding(src_vocab_size, emb_size)
        self.tgt_tok_emb = TokenEmbedding(tgt_vocab_size, emb_size)
        self.positional_encoding = PositionalEncoding(d_model= emb_size, max_len=max_len,
                                                      dropout=dropout)
        self.transformer = nn.Transformer(d_model=emb_size, nhead=nhead,
                                          num_encoder_layers=num_encoder_layers,
                                          num_decoder_layers=num_decoder_layers,
                                          dim_feedforward=dim_feedforward,
                                          dropout=dropout)
        self.generator = nn.Linear(emb_size, tgt_vocab_size)

    def forward(self, src,tgt,src_mask,tgt_mask,src_padding_mask,tgt_padding_mask,memory_key_padding_mask):
        src_emb = self.positional_encoding(self.src_tok_emb(src))
        tgt_emb = self.positional_encoding(self.tgt_tok_emb(tgt))
        outs = self.transformer(src = src_emb,
                                tgt = tgt_emb,
                                src_mask = src_mask,
                                tgt_mask = tgt_mask,
                                memory_mask=None,
                                src_key_padding_mask = src_padding_mask,
                                tgt_key_padding_mask = tgt_padding_mask,
                                memory_key_padding_mask = memory_key_padding_mask)
        return self.generator(outs)

    def encode(self, src, src_mask):
        return self.transformer.encoder(self.positional_encoding(self.src_tok_emb(src)),src_mask)

    def decode(self, tgt, memory, tgt_mask):
        return self.transformer.decoder(self.positional_encoding(self.tgt_tok_emb(tgt)),memory,tgt_mask)
    
    

### 11-6. 모델, 손실함수, 옵티마이저 생성

In [ ]:
from torch import optim

BATCH_SIZE = 128

model = Seq2SeqTransformer(num_encoder_layers = 3, num_decoder_layers=3,
                           emb_size=512,max_len=512,nhead=8,src_vocab_size=len(vocab_transform[SRC_LANG]),
                           tgt_vocab_size=len(vocab_transform[TGT_LANG]),
                           dim_feedforward= 512).to(device)

criterion = nn.CrossEntropyLoss(ignore_index = PAD_IDX).to(device)
optimizer = optim.Adam(model.parameters())

for main_name,main_module in model.named_children():
    print(main_name)
    for sub_name, sub_module in main_module.named_children():
        print("-",sub_name)
        for ssub_name, ssub_module in sub_module.named_children():
            print("--",ssub_name)
            for sssub_name, sssub_module in ssub_module.named_children():
                print("---",sssub_name)

### 11-7. 데이터 전처리 파이프라인 및 데이터로더 구성

In [ ]:
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

def sequential_transforms(*transforms):
    def func(txt_input):
        for transform in transforms:
            txt_input = transform(txt_input)
        return txt_input
    return func

def input_transform(token_ids):
    return torch.cat((torch.tensor([BOS_IDX]), torch.tensor(token_ids),
                      torch.tensor([EOS_IDX])))

def collator(batch):
    src_batch, tgt_batch = [], []
    for src_sample, tgt_sample in batch:
        src_batch.append(text_transform[SRC_LANG](src_sample.rstrip('\n')))
        tgt_batch.append(text_transform[TGT_LANG](tgt_sample.rstrip('\n')))

    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX)
    tgt_batch = pad_sequence(tgt_batch, padding_value=PAD_IDX)
    return src_batch, tgt_batch

text_transform = {}
for language in [SRC_LANG, TGT_LANG]:
    text_transform[language] = sequential_transforms(token_transform[language],
                                                     vocab_transform[language], input_transform)
data_iter = Multi30k(split='valid', language_pair=(SRC_LANG,TGT_LANG))
dataloader = DataLoader(data_iter, batch_size=BATCH_SIZE, collate_fn= collator)

source_tensor, target_tensor = next(iter(dataloader))

print(next(iter(data_iter)))
print(source_tensor.shape)
print(source_tensor)

### 11-8. 마스크(mask) 생성

In [ ]:
def generate_square_subsquent_mask(s):
    mask = (torch.triu(torch.ones((s,s),device=device))==1).transpose(0,1)
    mask = (mask.float().masked_fill(mask==0,float('-inf')).masked_fill(mask==1, float(0.0)))
    return mask

def create_mask(src, tgt):
    src_seq_len = src.shape[0]
    tgt_seq_len = tgt.shape[0]

    tgt_mask = generate_square_subsquent_mask(tgt_seq_len)
    src_mask = torch.zeros((src_seq_len, src_seq_len), device=device)

    src_padding_mask = (src==PAD_IDX).transpose(0,1)
    tgt_padding_mask = (tgt==PAD_IDX).transpose(0,1)
    return src_mask, tgt_mask, src_padding_mask, tgt_padding_mask

target_input = target_tensor[:-1,:]
target_output = target_tensor[1:,:]

source_mask, target_mask, source_padding_mask, target_padding_mask = create_mask(source_tensor, target_input)

print(source_mask.shape)
print(target_mask.shape)
print(source_padding_mask.shape)
print(target_padding_mask.shape)

### 11-9. 학습/검증 루프 정의 및 학습

In [ ]:
def run(model, optimizer, criterion, split):
    model.train() if split == 'train' else model.eval()
    data_iter = Multi30k(split= split, language_pair=(SRC_LANG,TGT_LANG))
    dataloader = DataLoader(data_iter,batch_size=BATCH_SIZE, collate_fn= collator)

    losses = 0
    for source_batch, target_batch in dataloader:
        source_batch=source_batch.to(device)
        target_batch = target_batch.to(device)

        target_input = target_batch[:-1,:]
        target_output = target_batch[1:,:]

        src_mask, tgt_mask, src_padding_mask, tgt_padding_mask = create_mask(source_batch, target_input)

        logits = model(src= source_batch, tgt = target_input, src_mask=src_mask,tgt_mask=tgt_mask,
                       src_padding_mask=src_padding_mask, tgt_padding_mask=tgt_padding_mask,
                       memory_key_padding_mask = src_padding_mask)
        optimizer.zero_grad()
        loss = criterion(logits.reshape(-1, logits.shape[-1]),target_output.reshape(-1))
        if split =='train':
            loss.backward()
            optimizer.step()
        losses += loss.item()
    return losses / len(list(dataloader))



In [ ]:
for epoch in range(5):
    train_loss = run(model, optimizer, criterion, 'train')
    val_loss = run(model, optimizer,criterion,'valid')
    print(epoch+1,train_loss,val_loss)

### 11-10. Greedy Decoding을 이용한 번역

In [ ]:
def greedy_decode(model, source_tensor, source_mask, max_len, start_symbol):
    source_tensor = source_tensor.to(device)
    source_mask = source_mask.to(device)

    memory = model.encode(source_tensor, source_mask)
    ys = torch.ones(1,1).fill_(start_symbol).type(torch.long).to(device)
    for i in range(max_len - 1):
        memory = memory.to(device)
        target_mask = generate_square_subsquent_mask(ys.size(0))
        target_mask = target_mask.type(torch.bool).to(device)

        out = model.decode(ys, memory, target_mask)
        out = out.transpose(0,1)
        prob = model.generator(out[:,-1])
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.item()

        ys= torch.cat([ys, torch.ones(1,1).type_as(source_tensor.data).fill_(next_word)],dim=0)

        if next_word == EOS_IDX:
            break

    return ys

def translate(model, source_sentence):
    model.eval()
    source_tensor = text_transform[SRC_LANG](source_sentence).view(-1,1)
    num_tokens = source_tensor.shape[0]
    src_mask = (torch.zeros(num_tokens,num_tokens)).type(torch.bool)
    tgt_tokens = greedy_decode(model, source_tensor, src_mask, max_len=num_tokens+5,
                               start_symbol=BOS_IDX).flatten()
    output = vocab_transform[TGT_LANG].lookup_tokens(list(tgt_tokens.cpu().numpy()))[1:-1]
    return " ".join(output)

output_oov = translate(model,"Eine Gruppe von Menschen steht vor einem Iglu.")
output = translate(model, "Eine Gruppe von Menschen steht vor einem Gebäude")
print(output_oov)
print(output)

## 12. 사전학습 언어모델 활용 (1) — GPT-2 텍스트 생성 & BERT 감성분류

### 12-1. GPT-2를 이용한 텍스트 생성 (`transformers.pipeline`)

In [ ]:
import torch
from transformers import pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

generator = pipeline(task='text-generation',model='gpt2',device=device)
outputs = generator(text_inputs = 'Machine learning is', max_length=28,
                    num_return_sequences=3,pad_token_id=generator.tokenizer.eos_token_id)
print(outputs)

### 12-2. NSMC 데이터 준비 (train/valid/test 분리)

In [ ]:
import numpy as np
import pandas as pd
from Korpora import Korpora

corpus = Korpora.load('nsmc')
df = pd.DataFrame(corpus.test).sample(20000,random_state=42)

In [ ]:
train, valid, test = np.split(
    df.sample(frac=1, random_state=42), 
    [int(0.6 * len(df)), int(0.8 * len(df))]
)

### 12-3. BERT 토크나이저로 데이터셋/데이터로더 구성

In [ ]:
import torch
from transformers import BertTokenizer
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

def make_dataset(data,tokenizer,device):
    tokenized = tokenizer(text=data.text.tolist(), padding='longest', truncation=True,return_tensors='pt')

    inputs_ids= tokenized['input_ids'].to(device)
    attention_mask = tokenized['attention_mask'].to(device)
    labels = torch.tensor(data.label.values, dtype=torch.long).to(device)
    return TensorDataset(inputs_ids, attention_mask, labels)

def get_dataloader(dataset, sampler, batch_size):
    data_sampler = sampler(dataset)
    dataloader = DataLoader(dataset, sampler=data_sampler,
                            batch_size = batch_size)
    return dataloader

epochs=5
batch_size=32

tokenizer = BertTokenizer.from_pretrained(pretrained_model_name_or_path='bert-base-multilingual-cased',
                                          do_lower_case=False)
train_dataset = make_dataset(train, tokenizer, device)
train_dataloader = get_dataloader(train_dataset, RandomSampler, batch_size)

valid_dataset = make_dataset(valid, tokenizer, device)
valid_dataloader = get_dataloader(valid_dataset, SequentialSampler, batch_size)

test_dataset = make_dataset(test, tokenizer, device)
test_dataloader = get_dataloader(test_dataset, SequentialSampler, batch_size)

print(train_dataset[0])

### 12-4. BERT 사전학습 모델(BertForSequenceClassification) 로드

In [ ]:
from torch import optim
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(pretrained_model_name_or_path='bert-base-multilingual-cased', num_labels=2).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5,eps=1e-8)

for main_name,main_module in model.named_children():
    print(main_name)
    for sub_name, sub_module in main_module.named_children():
        print("-",sub_name)
        for ssub_name, ssub_module in sub_module.named_children():
            print("--",ssub_name)
            for sssub_name, sssub_module in ssub_module.named_children():
                print("---",sssub_name)

### 12-5. 학습/평가 함수 정의 및 학습

In [ ]:
import numpy as np
from torch import nn

def calc_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat)/len(labels_flat)

def train(model, optimizer,dataloader):
    model.train()
    train_loss = 0.0

    for input_ids, attention_mask, labels in dataloader:
        outputs = model(input_ids=input_ids, attention_mask=attention_mask,
                        labels=labels)
        loss = outputs.loss
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss = train_loss/len(dataloader)
    return train_loss

def evaluation(model, dataloader):
    with torch.no_grad():
        model.eval()
        criterion = nn.CrossEntropyLoss()
        val_loss, val_accuracy = 0.0, 0.0
        for input_ids, attention_mask, labels in dataloader:
            outputs = model(input_ids=input_ids, attention_mask=attention_mask,
                            labels= labels)
            logits = outputs.logits
            loss = criterion(logits, labels)
            logits = logits.detach().cpu().numpy()
            label_ids = labels.to('cpu').numpy()
            accuracy = calc_accuracy(logits, label_ids)

            val_loss += loss.item()
            val_accuracy += accuracy

    val_loss = val_loss/len(dataloader)
    val_accuracy = val_accuracy/len(dataloader)
    return val_loss, val_accuracy

In [ ]:
best_loss = 10000
for epoch in range(epochs):
    train_loss = train(model, optimizer, train_dataloader)
    val_loss, val_accuracy = evaluation(model, valid_dataloader)
    print(epoch+1, train_loss, val_loss, val_accuracy)

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(),'./models/BertForSequenceClassification.pt')
        print('saved')

### 12-6. 저장된 모델로 테스트셋 평가

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path='bert-base-multilingual-cased',
    num_labels =2).to(device)
model.load_state_dict(torch.load('./models/BertForSequenceClassification.pt'))

test_loss, test_accuracy = evaluation(model, test_dataloader)
print(test_loss)
print(test_accuracy)

## 13. 사전학습 언어모델 활용 (2) — BART 뉴스 요약

영문 뉴스 요약 데이터셋(`argilla/news-summary`)으로 BART(`facebook/bart-base`) 모델을 파인튜닝해 요약 문장을 생성합니다.

### 13-1. 뉴스 요약 데이터셋 준비

In [ ]:
from datasets import load_dataset
import numpy as np
news = load_dataset('argilla/news-summary', split = 'test')
df = news.to_pandas().sample(5000,random_state=42)[['text','prediction']]
df['prediction'] = df['prediction'].map(lambda x: x[0]['text'])

train, valid, test = np.split(df.sample(frac=1,random_state=42),
                                        [int(0.6 * len(df)), int(0.8*len(df))])
print(train.text.iloc[0][:200])
print(train.prediction.iloc[0][:50])
print(len(train))
print(len(valid))
print(len(test))

### 13-2. BART 토크나이저로 데이터셋/데이터로더 구성

In [ ]:
import torch
from transformers import BartTokenizer
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.nn.utils.rnn import pad_sequence

def make_dataset(data, tokenizer, device):
    tokenized = tokenizer(text = data.text.tolist(), padding='longest',
                          truncation=True, return_tensors='pt', max_length=1024)
    labels = []
    input_ids = tokenized['input_ids'].to(device)
    attention_mask = tokenized['attention_mask'].to(device)
    for target in data.prediction:
        labels.append(tokenizer.encode(target, return_tensors='pt').squeeze())
    labels = pad_sequence(labels, batch_first=True, padding_value= -100).to(device)
    return TensorDataset(input_ids, attention_mask, labels)

def get_dataloader(dataset, sampler, batch_size):
    data_sampler = sampler(dataset)
    dataloader = DataLoader(dataset, sampler= data_sampler, batch_size=batch_size)
    return dataloader

epochs=5
batch_size=8

tokenizer = BartTokenizer.from_pretrained(pretrained_model_name_or_path='facebook/bart-base')

train_dataset = make_dataset(train, tokenizer, device)
train_dataloader = get_dataloader(train_dataset, RandomSampler, batch_size)


valid_dataset = make_dataset(valid, tokenizer, device)
valid_dataloader = get_dataloader(valid_dataset, SequentialSampler, batch_size)

test_dataset = make_dataset(test, tokenizer, device)
test_dataloader = get_dataloader(test_dataset, SequentialSampler, batch_size)

print(train_dataset[0])

### 13-3. BART 사전학습 모델(BartForConditionalGeneration) 로드

In [ ]:
from torch import optim
from transformers import BartForConditionalGeneration

model = BartForConditionalGeneration.from_pretrained(pretrained_model_name_or_path='facebook/bart-base').to(device)
optimizer = optim.AdamW(model.parameters(), lr = 5e-5, eps = 1e-8)


In [ ]:
for main_name,main_module in model.named_children():
    print(main_name)
    for sub_name, sub_module in main_module.named_children():
        print("-",sub_name)
        for ssub_name, ssub_module in sub_module.named_children():
            print("--",ssub_name)
            for sssub_name, sssub_module in ssub_module.named_children():
                print("---",sssub_name)

### 13-4. ROUGE 평가지표를 포함한 학습/평가 함수 정의

In [ ]:
import evaluate 

def calc_rouge(preds, labels):
    preds = preds.argmax(axis=-1)
    labels = np.where(labels !=-100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels,skip_special_tokens=True)

    rouge2 = rouge_score.compute(predictions = decoded_preds,
                                references = decoded_labels)
    return rouge2['rouge2']


def train(model,optimizer,dataloader):
    model.train()
    train_loss = 0.0

    for input_ids, attention_mask, labels in dataloader:
        outputs = model(input_ids=input_ids, attention_mask=attention_mask,
                        labels=labels)
        loss = outputs.loss
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss = train_loss / len(dataloader)
    return train_loss

def evaluation(model, dataloader):
    with torch.no_grad():
        model.eval()
        val_loss, val_rouge = 0.0, 0.0
        for input_ids, attention_mask, labels in dataloader:
            outputs = model(input_ids=input_ids,
                            attention_mask=attention_mask, labels=labels)
            logits = outputs.logits
            loss=outputs.loss

            logits = logits.detach().cpu().numpy()
            label_ids = labels.to('cpu').numpy()
            rouge = calc_rouge(logits, label_ids)

            val_loss += loss.item()
            val_rouge += rouge

    val_loss = val_loss / len(dataloader)
    val_rouge = val_rouge / len(dataloader)
    return val_loss, val_rouge



### 13-5. 학습 실행

In [ ]:
rouge_score = evaluate.load('rouge', tokenizer=tokenizer)
best_loss = 10000

for epoch in range(epochs):
    train_loss = train(model, optimizer, train_dataloader)
    val_loss, val_rouge = evaluation(model, valid_dataloader)
    print(epoch+1, train_loss, val_loss, val_rouge)

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(),'./models/BartForConditionalGeneration.pt')
        print('Saved!!')

### 13-6. 저장된 모델로 테스트셋 평가

In [ ]:
model = BartForConditionalGeneration.from_pretrained(pretrained_model_name_or_path='facebook/bart-base').to(device)
model.load_state_dict(torch.load('./models/BartForConditionalGeneration.pt'))

test_loss, test_rouge = evaluation(model, test_dataloader)
print(test_loss)
print(test_rouge)

### 13-7. 파이프라인으로 실제 요약 결과 확인

In [ ]:
from transformers import pipeline

summarizer = pipeline(task='summarization', model=model,
                      tokenizer=tokenizer,max_length=54,device='cpu')

for index in range(5):
    news_text = test.text.iloc[index]
    summarization = test.prediction.iloc[index]
    predicted_summarization = summarizer(news_text)[0]['summary_text']
    print(summarization)
    print(predicted_summarization)

## 14. 사전학습 언어모델 활용 (3) — ELECTRA 한국어 감성분류

한국어 사전학습 모델 `monologg/koelectra-base-v3-discriminator`로 NSMC 감성분류를 파인튜닝합니다.

### 14-1. NSMC 데이터 준비

In [ ]:
import numpy as np
import pandas as pd
from Korpora import Korpora

corpus = Korpora.load('nsmc')
df = pd.DataFrame(corpus.test).sample(20000, random_state=42)
train,valid,test=np.split(df.sample(frac=1,random_state=42),[int(0.6*len(df)),int(0.8*len(df))])

print(train.head().to_markdown())
print(len(train))
print(len(valid))
print(len(test))

### 14-2. ELECTRA 토크나이저로 데이터셋/데이터로더 구성 함수 정의

In [ ]:
import torch
from transformers import ElectraTokenizer
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

def make_dataset(data, tokenizer, device):
    tokenized = tokenizer(text=data.text.tolist(),padding='longest',
                          truncation=True, return_tensors ='pt')
    input_ids = tokenized['input_ids'].to(device)
    attention_mask = tokenized['attention_mask'].to(device)
    labels = torch.tensor(data.label.values,dtype=torch.long).to(device)
    return TensorDataset(input_ids, attention_mask, labels)

def get_dataloader(dataset, sampler, batch_size):
    data_sampler = sampler(dataset)
    dataloader = DataLoader(dataset, sampler=data_sampler,
                            batch_size = batch_size)
    return dataloader

### 14-3. 데이터로더 생성

In [ ]:
epochs=5
batch_size=32

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

tokenizer= ElectraTokenizer.from_pretrained(pretrained_model_name_or_path='monologg/koelectra-base-v3-discriminator',
                                            do_lower_cs=False)

train_dataset = make_dataset(train, tokenizer, device)
train_dataloader = get_dataloader(train_dataset, RandomSampler, batch_size)

valid_dataset = make_dataset(valid, tokenizer, device)
valid_dataloader = get_dataloader(valid_dataset, SequentialSampler, batch_size)

test_dataset = make_dataset(test, tokenizer, device)
test_dataloader = get_dataloader(test_dataset, SequentialSampler, batch_size)

print(train_dataset[0])



### 14-4. ELECTRA 사전학습 모델(ElectraForSequenceClassification) 로드

In [ ]:
from torch import optim
from transformers import ElectraForSequenceClassification

model = ElectraForSequenceClassification.from_pretrained(pretrained_model_name_or_path=
                                                         'monologg/koelectra-base-v3-discriminator'
                                                         , num_labels=2).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, eps = 1e-8)

for main_name,main_module in model.named_children():
    print(main_name)
    for sub_name, sub_module in main_module.named_children():
        print("-",sub_name)
        for ssub_name, ssub_module in sub_module.named_children():
            print("--",ssub_name)
            for sssub_name, sssub_module in ssub_module.named_children():
                print("---",sssub_name)

### 14-5. 학습/평가 함수 정의

In [ ]:
import numpy as np
from torch import nn

def calc_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

def train(model,optimizer,dataloader):
    model.train()
    train_loss = 0.0

    for input_ids, attention_mask, labels in dataloader:
        outputs = model(input_ids=input_ids, attention_mask=attention_mask,
                        labels=labels)
        loss = outputs.loss
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss = train_loss / len(dataloader)
    return train_loss

In [ ]:
def evaluation(model, dataloader):
    with torch.no_grad():
        model.eval()
        criterion = nn.CrossEntropyLoss()
        val_loss, val_accuracy = 0.0, 0.0
        for input_ids, attention_mask, labels in dataloader:
            outputs = model(input_ids=input_ids, attention_mask=attention_mask,
                            labels= labels)
            logits = outputs.logits
            loss = criterion(logits, labels)
            logits = logits.detach().cpu().numpy()
            label_ids = labels.to('cpu').numpy()
            accuracy = calc_accuracy(logits, label_ids)

            val_loss += loss.item()
            val_accuracy += accuracy

    val_loss = val_loss/len(dataloader)
    val_accuracy = val_accuracy/len(dataloader)
    return val_loss, val_accuracy

### 14-6. 학습 실행

In [ ]:
best_loss = 10000
for epoch in range(epochs):
    train_loss = train(model, optimizer, train_dataloader)
    val_loss, val_accuracy = evaluation(model, valid_dataloader)
    print(epoch+1, train_loss, val_loss, val_accuracy)

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(),'./models/ElectraForSequenceClassification.pt')
        print('saved')

### 14-7. 저장된 모델로 테스트셋 평가

In [ ]:
model = ElectraForSequenceClassification.from_pretrained(pretrained_model_name_or_path='monologg/koelectra-base-v3-discriminator',
                                                         num_labels=2).to(device)
model.load_state_dict(torch.load('./models/ElectraForSequenceClassification.pt'))
test_loss, test_accuracy=evaluation(model,test_dataloader)
print(test_loss)
print(test_accuracy)



## 15. 사전학습 언어모델 활용 (4) — T5 뉴스 요약

`t5-small` 모델을 이용해 (`summarize: ` 접두어를 붙인) 뉴스 요약 태스크를 인코더-디코더 구조로 파인튜닝합니다.

### 15-1. 뉴스 요약 데이터셋 준비 (`summarize:` 접두어 추가)

In [ ]:
import numpy as np
from datasets import load_dataset

news = load_dataset('argilla/news-summary', split='test')
df = news.to_pandas().sample(5000, random_state=42)[['text', 'prediction']]
df['text'] = 'summarize: ' + df['text']
df['prediction'] = df['prediction'].map(lambda x: x[0]['text'])
train, valid, test = np.split(df.sample(frac=1, random_state=42),
                              [int(0.6*len(df)),int(0.8*len(df))])
print(train.text.iloc[0][:200])
print(train.prediction.iloc[0][:50])
print(len(train))
print(len(valid))
print(len(test))

### 15-2. T5 토크나이저로 데이터셋/데이터로더 구성 함수 정의

In [ ]:
import torch
from transformers import T5Tokenizer
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.nn.utils.rnn import pad_sequence

def make_dataset(data,tokenizer,device):
    source = tokenizer(text = data.text.tolist(),padding='max_length',
                       max_length=128, pad_to_max_length=True, truncation=True,
                       return_tensors='pt')
    target = tokenizer(text = data.prediction.tolist(),padding='max_length',
                       max_length=128, pad_to_max_length=True, truncation=True,
                       return_tensors='pt')
    source_ids = source['input_ids'].squeeze().to(device)
    source_mask = source['attention_mask'].squeeze().to(device)
    target_ids = target['input_ids'].squeeze().to(device)
    target_mask = target['attention_mask'].squeeze().to(device)
    return TensorDataset(source_ids, source_mask,target_ids,target_mask)

def get_dataloader(dataset, sampler, batch_size):
    data_sampler = sampler(dataset)
    dataloader = DataLoader(dataset, sampler=data_sampler,
                            batch_size = batch_size)
    return dataloader

### 15-3. 데이터로더 생성

In [ ]:
epochs = 5
batch_size=8

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

tokenizer= T5Tokenizer.from_pretrained(pretrained_model_name_or_path='t5-small',
                                        )

train_dataset = make_dataset(train, tokenizer, device)
train_dataloader = get_dataloader(train_dataset, RandomSampler, batch_size)

valid_dataset = make_dataset(valid, tokenizer, device)
valid_dataloader = get_dataloader(valid_dataset, SequentialSampler, batch_size)

test_dataset = make_dataset(test, tokenizer, device)
test_dataloader = get_dataloader(test_dataset, SequentialSampler, batch_size)

print(next(iter(train_dataloader)))
print(tokenizer.convert_ids_to_tokens(21603))
print(tokenizer.convert_ids_to_tokens(10))


### 15-4. T5 사전학습 모델(T5ForConditionalGeneration) 로드

In [ ]:
from torch import optim
from transformers import T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained(pretrained_model_name_or_path='t5-small').to(device)

optimizer = optim.AdamW(model.parameters(), lr=1e-5,eps=1e-8)


### 15-5. 학습/평가 함수 정의

In [ ]:
import numpy as np
from torch import nn

def calc_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

def train(model, optimizer, dataloader):
    model.train()
    train_loss = 0.0

    for source_ids, source_mask, target_ids, target_mask in dataloader:
        decoder_input_ids = target_ids[:,:-1].contiguous()
        labels = target_ids[:, 1:].clone().detach()
        labels[target_ids[:,1:]==tokenizer.pad_token_id] = -100

        outputs = model(input_ids = source_ids, attention_mask=source_mask,
                        decoder_input_ids=decoder_input_ids,labels=labels)
        loss = outputs.loss
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss = train_loss / len(dataloader)
    return train_loss

def evaluation(model, dataloader):
    with torch.no_grad():
        model.eval()
        val_loss = 0.0

        for source_ids, source_mask, target_ids, target_mask in dataloader:
            decoder_input_ids = target_ids[:, :-1].contiguous()
            labels = target_ids[:,1:].clone().detach()
            labels[target_ids[:,1:] == tokenizer.pad_token_id] = -100

            outputs = model(input_ids = source_ids, attention_mask= source_mask,
                            decoder_input_ids=decoder_input_ids, labels=labels)

            loss = outputs.loss
            val_loss += loss.item()

        val_loss = val_loss / len(dataloader)
        return val_loss

### 15-6. 학습 실행

In [ ]:
best_loss = 10000
for epoch in range(epochs):
    train_loss = train(model, optimizer, train_dataloader)
    val_loss = evaluation(model, valid_dataloader)
    print(epoch+1, train_loss, val_loss)

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(),'./models/T5ForConditionalGeneration.pt')
        print('saved')

### 15-7. 테스트셋에 대한 생성 결과 확인 (beam search 디코딩)

In [ ]:
model.eval()
with torch.no_grad():
    for source_ids, source_mask, target_ids, target_mask in test_dataloader:
        generated_ids = model.generate(input_ids=source_ids,
                                       attention_mask=source_mask,max_length=128,
                                       num_beams=3,repetition_penalty=2.5,
                                       length_penalty=1.0, early_stopping=True)
        for generated, target in zip(generated_ids, target_ids):
            pred = tokenizer.decode(generated,skip_special_tokens=True,
                                    clean_up_tokenization_spaces=True)
            actual = tokenizer.decode(target, skip_special_tokens=True,
                                      clean_up_tokenization_spaces=True)
            print(pred)
            print(actual)
        break
